# Classificador de Spam (NLP Básico)

Este notebook mostra, passo a passo, como treinar um classificador supervisionado para prever se uma mensagem é **spam** ou **ham** (não spam).

## O que você vai aprender
- Como carregar e entender o dataset `SMS Spam Collection`.
- Como limpar texto com NLTK.
- Como transformar texto em números com TF-IDF.
- Como treinar `MultinomialNB` e `LogisticRegression`.
- Como avaliar os modelos de forma legível.

> Objetivo didático: mostrar claramente os dados antes e depois do pré-processamento para análise humana antes do treino.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# display() funciona no Jupyter; fora dele, usamos print como fallback
try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root(start: Path, max_levels: int = 8) -> Path:
    """Sobe diretórios a partir de cwd até achar src/load_data.py."""
    cur = start.resolve()
    for _ in range(max_levels):
        marker = cur / "src" / "load_data.py"
        if marker.is_file():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError(
        "Não achei a raiz do projeto (arquivo src/load_data.py). "
        "Abra o Jupyter/VS Code na pasta do projeto ou rode: cd .../classify_messages_as_spam_or_not_spam"
    )


# cwd pode ser a pasta do projeto, notebooks/, ou outro subdiretório
PROJECT_ROOT = find_project_root(Path.cwd())

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from load_data import default_raw_path, download_sms_spam_collection, load_sms_spam
from preprocess import ensure_nltk_data, preprocess_series

pd.set_option("display.max_colwidth", 160)
PROJECT_ROOT

## 1) Configuração dos hiperparâmetros

Nesta célula você controla os principais parâmetros do experimento. Como é um notebook didático, deixar tudo explícito ajuda a reproduzir resultados.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_FEATURES = 10_000
REMOVE_STOPWORDS = True

print({
    "RANDOM_STATE": RANDOM_STATE,
    "TEST_SIZE": TEST_SIZE,
    "MAX_FEATURES": MAX_FEATURES,
    "REMOVE_STOPWORDS": REMOVE_STOPWORDS,
})

## 2) Carregar dataset

Aqui carregamos o `SMS Spam Collection`. Se o arquivo não existir em `data/raw/`, ele é baixado automaticamente da UCI.

In [ ]:
raw_path = default_raw_path()
if not raw_path.is_file():
    raw_path = download_sms_spam_collection()

df = load_sms_spam(raw_path)
print(f"Dataset path: {raw_path}")
print(f"Shape: {df.shape}")
df.head(10)

## 3) Análise inicial do DataFrame (dados brutos)

Antes de treinar, analisamos os dados para entender:
- distribuição das classes (spam vs ham),
- tamanho médio das mensagens,
- qualidade geral dos textos.

In [ ]:
analysis_df = df.copy()
analysis_df["text_length"] = analysis_df["text"].str.len()
analysis_df["token_count_raw"] = analysis_df["text"].str.split().str.len()

print("Class distribution (0=ham, 1=spam):")
display(analysis_df["label"].value_counts().to_frame("count"))

print("Length stats by class:")
display(
    analysis_df.groupby("label")["text_length"].describe()[["mean", "std", "min", "50%", "max"]]
)

print("Raw sample:")
display(analysis_df[["label", "text", "text_length", "token_count_raw"]].head(8))

## 4) Pré-processamento de texto (NLTK)

Aplicamos limpeza para padronizar as mensagens:
- lowercase,
- tokenização,
- remoção de ruído,
- lematização,
- remoção opcional de stopwords.

O objetivo é preparar os dados para o TF-IDF.

In [ ]:
ensure_nltk_data()

analysis_df["text_clean"] = preprocess_series(
    analysis_df["text"].tolist(),
    remove_stopwords=REMOVE_STOPWORDS,
)
analysis_df["token_count_clean"] = analysis_df["text_clean"].str.split().str.len()

display(
    analysis_df[["label", "text", "text_clean", "text_length", "token_count_raw", "token_count_clean"]].head(12)
)

print("Exemplo de mensagens SPAM após limpeza:")
display(analysis_df.loc[analysis_df["label"] == 1, ["text", "text_clean"]].head(5))

print("Exemplo de mensagens HAM após limpeza:")
display(analysis_df.loc[analysis_df["label"] == 0, ["text", "text_clean"]].head(5))

## 5) Divisão treino/teste

Agora separamos os dados em:
- **treino**: usado para aprender,
- **teste**: usado para validar desempenho em dados não vistos.

Usamos `stratify` para manter a proporção de classes nas duas partes.

In [ ]:
X = analysis_df["text_clean"].tolist()
y = analysis_df["label"].values

# X_train e X_test = entradas (features)
# y_train e y_test = saídas (labels)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "samples": [len(X_train), len(X_test)],
        "spam_ratio": [float((y_train == 1).mean()), float((y_test == 1).mean())],
    }
)
display(split_df)

## 6) Vetorização TF-IDF

Nesta etapa transformamos texto em números. Cada mensagem vira um vetor de pesos TF-IDF.

Configuração usada:
- `ngram_range=(1,2)` para unigramas e bigramas,
- `max_features` para limitar vocabulário,
- `min_df=2` para reduzir termos raríssimos.

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=MAX_FEATURES,
    min_df=2,
    sublinear_tf=True,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("X_train_vec shape:", X_train_vec.shape)
print("X_test_vec shape:", X_test_vec.shape)

feature_names = vectorizer.get_feature_names_out()
preview_features = pd.DataFrame({"feature": feature_names[:25]})
preview_features

## 7) Treinar modelos: Naive Bayes e Regressão Logística

Treinamos dois classificadores clássicos para comparar:
- `MultinomialNB` (baseline rápido para texto)
- `LogisticRegression` (modelo linear forte em TF-IDF).

**Importante:** execute o notebook **do início ao fim** (*Run All*) ou, após reiniciar o kernel, rode primeiro a **célula de imports** no topo. Se você rodar só esta célula num kernel novo, pode aparecer `NameError: MultinomialNB is not defined` porque os imports ainda não foram executados. Também é preciso ter rodado as células que criam `X_train_vec`, `X_test_vec`, `y_train` e `y_test`.

In [ ]:
# Imports aqui também: evita NameError se esta célula rodar sem a célula do topo
# (o ideal é executar o notebook desde o início).
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)

lr = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

print("Treino concluído.")

## 8) Avaliação comparativa (saída legível)

Aqui comparamos os dois modelos com:
- acurácia,
- precision/recall/F1,
- matriz de confusão.

Foco importante: **recall da classe spam** (quantos spams o modelo conseguiu capturar).

In [ ]:
# --- Como interpretar esta seção ---
# Tabela resumo:
#   accuracy       → % de previsões corretas no teste (ham + spam). Pode ser alta mesmo com classes desbalanceadas.
#   spam_precision → Entre tudo que o modelo marcou como spam, quanto era spam de verdade.
#                    Alta = poucos "falsos alarmes" (ham classificado como spam).
#   spam_recall    → Entre todos os spams reais no teste, quantos o modelo encontrou.
#                    Alto = pouco spam "passando" como ham (falso negativo).
#   spam_f1        → Equilíbrio entre precision e recall na classe spam (comparar modelos com um único número).
#
# classification_report (por classe):
#   precision/recall/f1 para ham e spam; support = quantidade real de exemplos da classe no teste.
#
# Matriz de confusão (linhas = verdade, colunas = previsão):
#   true_ham  × pred_ham  → acerto em ham
#   true_ham  × pred_spam → ham marcado como spam (erro: falso positivo de spam)
#   true_spam × pred_ham  → spam não detectado (erro: falso negativo de spam)
#   true_spam × pred_spam → acerto em spam
#
metrics_table = pd.DataFrame(
    {
        "model": ["MultinomialNB", "LogisticRegression"],
        "accuracy": [
            accuracy_score(y_test, y_pred_nb),
            accuracy_score(y_test, y_pred_lr),
        ],
        "spam_recall": [
            classification_report(y_test, y_pred_nb, output_dict=True)["1"]["recall"],
            classification_report(y_test, y_pred_lr, output_dict=True)["1"]["recall"],
        ],
        "spam_precision": [
            classification_report(y_test, y_pred_nb, output_dict=True)["1"]["precision"],
            classification_report(y_test, y_pred_lr, output_dict=True)["1"]["precision"],
        ],
        "spam_f1": [
            classification_report(y_test, y_pred_nb, output_dict=True)["1"]["f1-score"],
            classification_report(y_test, y_pred_lr, output_dict=True)["1"]["f1-score"],
        ],
    }
).sort_values("spam_f1", ascending=False)

display(metrics_table)

print("Classification report - MultinomialNB")
print(classification_report(y_test, y_pred_nb, target_names=["ham", "spam"]))
print("Confusion matrix - MultinomialNB")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_nb), index=["true_ham", "true_spam"], columns=["pred_ham", "pred_spam"]))

print("Classification report - LogisticRegression")
print(classification_report(y_test, y_pred_lr, target_names=["ham", "spam"]))
print("Confusion matrix - LogisticRegression")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_lr), index=["true_ham", "true_spam"], columns=["pred_ham", "pred_spam"]))


## 9) (Opcional) Salvar artefatos e versionar

Você pode **não** salvar (só experimentar), salvar no layout **legado** `models/*.joblib`, ou salvar em **`models/versions/vNN/`** com `metadata.json`.

- `SAVE_MODEL`: grava ou não os arquivos.
- `VERSION`: `None` usa o próximo nome livre (`v01`, `v02`, …); ou fixe, por exemplo, `"v03"`.
- `SET_CURRENT`: se `True`, grava `models/current` para o `predict.py` carregar essa versão por padrão.
- `LEGACY_FLAT`: se `True`, ignora versionamento e grava na raiz `models/` (compatível com fluxo antigo).

**Pré-requisito:** execute as células de treino e avaliação antes desta (usa `y_test`, `y_pred_nb`, `y_pred_lr`, `df`, `raw_path`).


In [ ]:
import joblib
from pathlib import Path

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from model_registry import (
    build_metadata,
    file_sha256,
    next_version_name,
    save_version_bundle,
    sklearn_params,
    validate_version,
)

SAVE_MODEL = True
VERSION = None  # ex.: "v03", ou None para auto-incremento
SET_CURRENT = False
LEGACY_FLAT = False

if not SAVE_MODEL:
    print("SAVE_MODEL=False: nada foi salvo em disco.")
elif LEGACY_FLAT:
    mdir = PROJECT_ROOT / "models"
    mdir.mkdir(parents=True, exist_ok=True)
    joblib.dump(vectorizer, mdir / "tfidf_vectorizer.joblib")
    joblib.dump(nb, mdir / "multinomial_nb.joblib")
    joblib.dump(lr, mdir / "logistic_regression.joblib")
    joblib.dump({"remove_stopwords": REMOVE_STOPWORDS}, mdir / "preprocess_config.joblib")
    print(f"Artefatos salvos (layout legado) em: {mdir}")
else:
    version = VERSION if VERSION is not None else next_version_name()
    validate_version(version)
    ds_path = Path(raw_path).resolve()
    sha = file_sha256(ds_path) if ds_path.is_file() else None
    metrics_for_meta = {
        "naive_bayes": {
            "accuracy": float(accuracy_score(y_test, y_pred_nb)),
            "report": classification_report(y_test, y_pred_nb, target_names=["ham", "spam"]),
            "confusion_matrix": confusion_matrix(y_test, y_pred_nb).tolist(),
        },
        "logistic_regression": {
            "accuracy": float(accuracy_score(y_test, y_pred_lr)),
            "report": classification_report(y_test, y_pred_lr, target_names=["ham", "spam"]),
            "confusion_matrix": confusion_matrix(y_test, y_pred_lr).tolist(),
        },
    }
    hyperparameters = {
        "train_test_split": {
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
            "stratify": True,
        },
        "max_features": MAX_FEATURES,
        "remove_stopwords": REMOVE_STOPWORDS,
        "tfidf_vectorizer": sklearn_params(vectorizer),
        "multinomial_nb": sklearn_params(nb),
        "logistic_regression": sklearn_params(lr),
    }
    metadata = build_metadata(
        version=version,
        dataset_path=str(ds_path),
        dataset_sha256=sha,
        n_rows=len(df),
        hyperparameters=hyperparameters,
        metrics=metrics_for_meta,
        parent_version=None,
        notes="notebook",
    )
    bundle = {
        "vectorizer": vectorizer,
        "nb": nb,
        "lr": lr,
        "preprocess_config": {"remove_stopwords": REMOVE_STOPWORDS},
    }
    dest = save_version_bundle(
        bundle,
        version=version,
        set_current=SET_CURRENT,
        metadata=metadata,
    )
    print(f"Artefatos salvos em: {dest}")
    if SET_CURRENT:
        print(f"models/current -> {version}")


## 9b) (Opcional) Perfil simples do texto (sanidade / "drift" leve)

Compare estatísticas de comprimento e contagem de tokens entre o **texto bruto** do treino e, se quiser, um **lote novo**. Não substitui monitoramento de modelo; só ajuda a ver se um novo batch é muito diferente em tamanho.


In [ ]:
from data_profile import profile_texts

print("Perfil — textos brutos (treino completo):")
display(profile_texts(df["text"]))

# Exemplo: descomente e passe uma lista de strings novas para comparar
# novos = ["short msg", "another one..."]
# display(profile_texts(novos))


## 11) Seção opcional: migração para embeddings (BERT)

Quando você quiser avançar além de TF-IDF:

1. Instale dependências (`sentence-transformers`, `torch`).
2. Use o script já pronto `src/train_bert_extension.py` para comparar com o baseline.

No terminal:

```bash
python src/train_bert_extension.py
```

Ou apenas embeddings (sem reimprimir baseline):

```bash
python src/train_bert_extension.py --skip-tfidf-baseline
```

Essa etapa é mais custosa computacionalmente, mas permite capturar contexto semântico melhor que bag-of-words em muitos cenários.

## 12) Conclusão

Foi executado uma pipeline completa de NLP clássico supervisionado:

- dados brutos carregados e inspecionados,
- dados limpos e comparados com o original,
- treino e teste separados corretamente,
- vetorização TF-IDF,
- treino de dois modelos,
- avaliação comparativa com foco em qualidade para spam.

Próximo passo recomendado: testar pequenas variações de hiperparâmetros e comparar o impacto no `spam_recall` e `spam_f1`.